# Group-wise Comparisons and Differential Expression

Compare gene expression and cell type composition across:
- Different conditions/treatments
- Different samples
- Different spatial regions

**Input:** Annotated AnnData with group labels

**Output:** Differential expression results and statistical comparisons

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sc.settings.set_figure_params(dpi=80)
print(f"Scanpy version: {sc.__version__}")

In [ ]:
# Load data
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures/04_group_comparisons")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = "xenium_sample_01"
adata = sc.read_h5ad(DATA_DIR / f"{SAMPLE_NAME}_spatial_analysis.h5ad")

print(f"Data shape: {adata.shape}")

## 1. Define Comparison Groups

Define groups for comparison (condition, treatment, region, etc.)

In [ ]:
# Example: Add group labels if not already present
# This should be customized based on your experimental design

# Option 1: If you have metadata with conditions
# adata.obs['condition'] = ...

# Option 2: Define groups based on spatial domains
if 'spatial_domain' in adata.obs:
    adata.obs['comparison_group'] = adata.obs['spatial_domain']
else:
    # Create example groups for demonstration
    adata.obs['comparison_group'] = 'Group_A'
    adata.obs.loc[adata.obs.index[len(adata)//2:], 'comparison_group'] = 'Group_B'

print("\nComparison groups:")
print(adata.obs['comparison_group'].value_counts())

## 2. Cell Type Composition Analysis

In [ ]:
# Calculate cell type proportions per group
composition = pd.crosstab(
    adata.obs['comparison_group'],
    adata.obs['celltype'],
    normalize='index'
) * 100

print("\nCell type composition (%) by group:")
print(composition)

# Plot composition
composition.T.plot(kind='bar', figsize=(12, 6))
plt.ylabel('Percentage')
plt.xlabel('Cell Type')
plt.title('Cell Type Composition by Group')
plt.legend(title='Group')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'celltype_composition.png', dpi=300, bbox_inches='tight')
plt.show()

# Save composition data
composition.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_celltype_composition.csv")

## 3. Statistical Testing of Cell Type Abundance

In [ ]:
# Chi-square test for cell type distribution
from scipy.stats import chi2_contingency

ct_counts = pd.crosstab(adata.obs['comparison_group'], adata.obs['celltype'])
chi2, pval, dof, expected = chi2_contingency(ct_counts)

print(f"\nChi-square test for cell type distribution:")
print(f"  Chi-square statistic: {chi2:.4f}")
print(f"  P-value: {pval:.4e}")
print(f"  Degrees of freedom: {dof}")

if pval < 0.05:
    print("  Result: Significant difference in cell type distribution")
else:
    print("  Result: No significant difference in cell type distribution")

## 4. Differential Expression Analysis

In [ ]:
# Perform DE between groups (overall)
sc.tl.rank_genes_groups(
    adata,
    groupby='comparison_group',
    method='wilcoxon',
    key_added='de_group',
    use_raw=False
)

# Plot top DE genes
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, key='de_group')
plt.savefig(FIGURES_DIR / 'de_genes_by_group.png', dpi=300, bbox_inches='tight')
plt.show()

# Extract DE results
de_results = sc.get.rank_genes_groups_df(adata, group=None, key='de_group')
de_results.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_de_genes_by_group.csv", index=False)
print(f"\nDE results saved with {len(de_results)} entries")

## 5. Cell Type-Specific Differential Expression

In [ ]:
# Perform DE within each cell type
de_celltype_results = []

for celltype in adata.obs['celltype'].unique():
    print(f"\nAnalyzing {celltype}...")
    
    # Subset to specific cell type
    adata_ct = adata[adata.obs['celltype'] == celltype].copy()
    
    # Check if we have both groups
    if adata_ct.obs['comparison_group'].nunique() < 2:
        print(f"  Skipping {celltype}: not enough groups")
        continue
    
    # Check minimum cell count
    if adata_ct.n_obs < 10:
        print(f"  Skipping {celltype}: too few cells")
        continue
    
    # Run DE
    sc.tl.rank_genes_groups(
        adata_ct,
        groupby='comparison_group',
        method='wilcoxon',
        key_added='de'
    )
    
    # Extract results
    ct_de = sc.get.rank_genes_groups_df(adata_ct, group=None, key='de')
    ct_de['celltype'] = celltype
    de_celltype_results.append(ct_de)
    
    print(f"  Found {len(ct_de)} DE genes")

# Combine results
if de_celltype_results:
    all_de_ct = pd.concat(de_celltype_results, ignore_index=True)
    all_de_ct.to_csv(
        OUTPUT_DIR / f"{SAMPLE_NAME}_de_genes_by_celltype.csv",
        index=False
    )
    print(f"\nCell type-specific DE results saved: {len(all_de_ct)} total entries")

## 6. Volcano Plots

In [ ]:
# Create volcano plot for overall DE
de_data = de_results.copy()

# Calculate -log10(pval)
de_data['-log10_pval'] = -np.log10(de_data['pvals_adj'] + 1e-300)

# Set significance thresholds
pval_thresh = 0.05
fc_thresh = 1.0

# Create volcano plot
fig, ax = plt.subplots(figsize=(10, 8))

# Plot non-significant genes
non_sig = de_data[
    (de_data['pvals_adj'] >= pval_thresh) |
    (np.abs(de_data['logfoldchanges']) < fc_thresh)
]
ax.scatter(non_sig['logfoldchanges'], non_sig['-log10_pval'],
           alpha=0.3, s=10, c='gray', label='Not significant')

# Plot significant genes
sig = de_data[
    (de_data['pvals_adj'] < pval_thresh) &
    (np.abs(de_data['logfoldchanges']) >= fc_thresh)
]
ax.scatter(sig['logfoldchanges'], sig['-log10_pval'],
           alpha=0.6, s=10, c='red', label='Significant')

# Add labels for top genes
top_genes = sig.nlargest(10, '-log10_pval')
for _, gene in top_genes.iterrows():
    ax.annotate(gene['names'], (gene['logfoldchanges'], gene['-log10_pval']),
                fontsize=8, alpha=0.7)

# Formatting
ax.axhline(-np.log10(pval_thresh), color='blue', linestyle='--', alpha=0.5)
ax.axvline(-fc_thresh, color='blue', linestyle='--', alpha=0.5)
ax.axvline(fc_thresh, color='blue', linestyle='--', alpha=0.5)
ax.set_xlabel('Log2 Fold Change')
ax.set_ylabel('-Log10 Adjusted P-value')
ax.set_title('Differential Expression Volcano Plot')
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'volcano_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSignificant DE genes: {len(sig)}")

## 7. Heatmap of Top DE Genes

In [ ]:
# Get top DE genes
n_top = 50
top_de_genes = de_results.nsmallest(n_top, 'pvals_adj')['names'].tolist()

# Plot heatmap
sc.pl.heatmap(
    adata,
    top_de_genes,
    groupby='comparison_group',
    swap_axes=True,
    cmap='RdBu_r',
    standard_scale='var',
    figsize=(8, 10)
)
plt.savefig(FIGURES_DIR / 'de_genes_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Gene Set Enrichment (if applicable)

In [ ]:
# Export gene lists for external enrichment analysis
# (e.g., using Enrichr, GSEA, or other tools)

# Get upregulated and downregulated genes
sig_genes = de_results[de_results['pvals_adj'] < 0.05]

upregulated = sig_genes[sig_genes['logfoldchanges'] > 1]['names'].tolist()
downregulated = sig_genes[sig_genes['logfoldchanges'] < -1]['names'].tolist()

# Save gene lists
with open(OUTPUT_DIR / f"{SAMPLE_NAME}_upregulated_genes.txt", 'w') as f:
    f.write('\n'.join(upregulated))

with open(OUTPUT_DIR / f"{SAMPLE_NAME}_downregulated_genes.txt", 'w') as f:
    f.write('\n'.join(downregulated))

print(f"Upregulated genes: {len(upregulated)}")
print(f"Downregulated genes: {len(downregulated)}")
print(f"\nGene lists saved for external enrichment analysis")

## 9. Summary Report

In [ ]:
# Generate summary
summary = {
    'Total cells': adata.n_obs,
    'Comparison groups': adata.obs['comparison_group'].nunique(),
    'Cell types': adata.obs['celltype'].nunique(),
    'Total DE genes (p<0.05)': len(de_results[de_results['pvals_adj'] < 0.05]),
    'Upregulated genes': len(upregulated),
    'Downregulated genes': len(downregulated),
}

summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['Value']

print("\n=== Group Comparison Summary ===")
print(summary_df)

summary_df.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_comparison_summary.csv")